In [4]:
import pandas as pd
from mplsoccer.pitch import Pitch
import matplotlib.pyplot as plt
from matplotlib.pyplot import figure
import numpy

wsl = pd.read_hdf('data-opta-woman-all/spadl-opta.h5', "games")
wsl_game_ids = wsl['game_id'].values
wsl_game_ids

array([1077833, 1077836, 1077837, 1077835, 1077834, 1077890, 1077884,
       1077853, 1077847, 1077927, 1077933, 1077932, 1077926, 1077852,
       1077885, 1077891, 1077893, 1077844, 1077850, 1077930, 1077924,
       1077925, 1077931, 1077851, 1077845, 1077879, 1077892, 1077886,
       1077882, 1077896, 1077841, 1077855, 1077869, 1077909, 1077908,
       1077934, 1077868, 1077854, 1077840, 1077897, 1077883, 1077895,
       1077881, 1077856, 1077842, 1077923, 1077843, 1077857, 1077880,
       1077894, 1077899, 1077872, 1077866, 1077906, 1077912, 1077907,
       1077867, 1077873, 1077859, 1077865, 1077871, 1077911, 1077905,
       1077904, 1077910, 1077870, 1077864, 1077858, 1077860, 1077874,
       1077848, 1077914, 1077900, 1077928, 1077915, 1077849, 1077875,
       1077861, 1077877, 1077863, 1077903, 1077902, 1077862, 1077876,
       1077889, 1077839, 1077838, 2138467, 2138473, 2138498, 2138499,
       2138472, 2138466, 2138470, 2138464, 2138458, 2138459, 2138465,
       2138471, 2138

In [5]:
wsl_matches = []

for id in wsl_game_ids:
    wsl_matches.append(pd.read_hdf('data-opta-woman-all/features.h5', f"game_{id}"))

wsl_matches = pd.concat(wsl_matches, ignore_index=True)
wsl_matches

,type_id_a0,type_id_a1,type_id_a2,type_pass_a0,type_cross_a0,type_throw_in_a0,type_freekick_crossed_a0,type_freekick_short_a0,type_corner_crossed_a0,type_corner_short_a0,...,time_seconds_a0,time_seconds_overall_a0,period_id_a1,time_seconds_a1,time_seconds_overall_a1,period_id_a2,time_seconds_a2,time_seconds_overall_a2,time_delta_1,time_delta_2
0,0,0,0,True,False,False,False,False,False,False,...,1.0,1.0,1,1.0,1.0,1,1.0,1.0,0.0,0.0
1,21,0,0,False,False,False,False,False,False,False,...,2.5,2.5,1,1.0,1.0,1,1.0,1.0,1.5,1.5
2,0,21,0,True,False,False,False,False,False,False,...,4.0,4.0,1,2.5,2.5,1,1.0,1.0,1.5,3.0
3,0,0,21,True,False,False,False,False,False,False,...,5.0,5.0,1,4.0,4.0,1,2.5,2.5,1.0,2.5
4,0,0,0,True,False,False,False,False,False,False,...,8.0,8.0,1,5.0,5.0,1,4.0,4.0,3.0,4.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
492680,0,0,0,True,False,False,False,False,False,False,...,3054.0,5754.0,2,3050.0,5750.0,2,3046.0,5746.0,4.0,8.0
492681,0,0,0,True,False,False,False,False,False,False,...,3059.0,5759.0,2,3054.0,5754.0,2,3050.0,5750.0,5.0,9.0
492682,21,0,0,False,False,False,False,False,False,False,...,3063.0,5763.0,2,3059.0,5759.0,2,3054.0,5754.0,4.0,9.0
492683,1,21,0,False,True,False,False,False,False,False,...,3067.0,5767.0,2,3063.0,5763.0,2,3059.0,5759.0,4.0,8.0


In [6]:
wsl_matches.rename(columns = {'start_x_a0':'start_x', 'start_y_a0':'start_y', 'end_x_a0':'end_x', 'end_y_a0':'end_y'}, inplace = True)  # type: ignore
locations = [(x, y) for x, y in zip(wsl_matches['start_x'], wsl_matches['start_y'])]
wsl_matches['location'] = locations

In [7]:
wsl_matches['start_x'].max()

105.0

In [12]:
wsl_matches['start_y'].max() / 8

8.5

In [13]:
grid_Xs = ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z']
grid_Ys = [i+1 for i in range(20)]

def get_area_x(loc_x, areas_x):
    for i in range (0,areas_x+1):
        if loc_x <= (105/areas_x)*i and loc_x > (105/areas_x)*(i-1):
            return grid_Xs[i-1]
    return "Error in get_area_x!"

def get_area_y(loc_y, areas_y):
    for i in range (0, areas_y+1):
        if loc_y <= (areas_y*8.5) - (i * 8.5) and loc_y > ((areas_y*8.5) - 8.5) - (i * 8.5):
            return grid_Ys[i]
    return "Error in get_area_y!"

def get_event_area(location, areas_x, areas_y):
    x = location[0]
    y = location[1]
    return f'{get_area_x(x, areas_x)}{get_area_y(y, areas_y)}'

def add_grid_areas(df, x_areas = 12, y_areas = 8):
    event_start_areas = [get_event_area([start_x, start_y], x_areas, y_areas) 
                         if start_x != '' or start_y != '' else '' for start_x, start_y in zip(df['start_x'].values, df['start_y'].values)]
    event_end_areas = [get_event_area([end_x, end_y], x_areas, y_areas) 
                       if end_x != '' or end_y != '' else '' for end_x, end_y in zip(df['end_x'].values, df['end_y'].values)]
    event_area_codes = [f'{x}{y}' for x,y in zip(event_start_areas, event_end_areas)]
    df['start_area'] = event_start_areas
    df['end_area'] = event_end_areas
    df['area_code'] = event_area_codes
    return df

In [14]:
wsl_matches = add_grid_areas(wsl_matches)
wsl_matches

,type_id_a0,type_id_a1,type_id_a2,type_pass_a0,type_cross_a0,type_throw_in_a0,type_freekick_crossed_a0,type_freekick_short_a0,type_corner_crossed_a0,type_corner_short_a0,...,time_seconds_overall_a1,period_id_a2,time_seconds_a2,time_seconds_overall_a2,time_delta_1,time_delta_2,location,start_area,end_area,area_code
0,0,0,0,True,False,False,False,False,False,False,...,1.0,1,1.0,1.0,0.0,0.0,"(52.5, 34.0)",F5,F4,F5F4
1,21,0,0,False,False,False,False,False,False,False,...,1.0,1,1.0,1.0,1.5,1.5,"(45.045, 35.428000000000004)",F4,E5,F4E5
2,0,21,0,True,False,False,False,False,False,False,...,2.5,1,1.0,1.0,1.5,3.0,"(40.53, 30.94)",E5,D6,E5D6
3,0,0,21,True,False,False,False,False,False,False,...,4.0,1,2.5,2.5,1.0,2.5,"(28.980000000000004, 25.092)",D6,D3,D6D3
4,0,0,0,True,False,False,False,False,False,False,...,5.0,1,4.0,4.0,3.0,4.0,"(26.775000000000002, 49.232000000000006)",D3,E1,D3E1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
492680,0,0,0,True,False,False,False,False,False,False,...,5750.0,2,3046.0,5746.0,4.0,8.0,"(49.455000000000005, 23.664)",F6,F3,F6F3
492681,0,0,0,True,False,False,False,False,False,False,...,5754.0,2,3050.0,5750.0,5.0,9.0,"(65.73, 6.867999999999999)",H8,J8,H8J8
492682,21,0,0,False,False,False,False,False,False,False,...,5759.0,2,3054.0,5754.0,4.0,9.0,"(79.065, 1.1560000000000001)",J8,L8,J8L8
492683,1,21,0,False,True,False,False,False,False,False,...,5763.0,2,3059.0,5759.0,4.0,8.0,"(102.69, 7.684)",L8,L7,L8L7
